# Posture Classification from 2D Human Pose — Demo

Detect people with **Faster R-CNN** (MMDetection), estimate 17 COCO keypoints with a
**SimpleBaseline ResNet-50** top-down model (MMPose), then label each person as
**Standing**, **Sitting**, **Lying** or **Unknown** with transparent geometric rules.

All reusable logic lives in the `posture_classifier` package; this notebook only
wires it together and shows the output.

**Contents**
1. [Setup](#1-setup)
2. [Download model weights](#2-download-model-weights)
3. [Run the pipeline](#3-run-the-pipeline)
4. [How the rules decide](#4-how-the-rules-decide)
5. [Limitations](#5-limitations)

## 1. Setup

Run from the repository root. The OpenMMLab stack needs a PyTorch build that `mmcv`
has pre-built wheels for (e.g. Python 3.10 + torch 2.1.x); see the README for details.
Skip the install cell if your environment is already set up.

In [ ]:
# Uncomment on a fresh machine / Colab runtime.
# !git clone https://github.com/<your-username>/pose-posture-classifier.git
# %cd pose-posture-classifier
# %pip install -U openmim
# !mim install -r requirements-mmlab.txt
# %pip install -e .

## 2. Download model weights

Configs and checkpoints (Apache-2.0, published by OpenMMLab) are fetched into
`checkpoints/`, which is git-ignored.

In [ ]:
# !bash scripts/download_models.sh

## 3. Run the pipeline

Set `IMAGE_PATH` to any photo with people in it. Use your own images and
do not commit photos of people without their consent.

In [ ]:
import logging
from pathlib import Path

import cv2
from IPython.display import Image, display

from posture_classifier.cli import person_to_dict
from posture_classifier.config import load_config
from posture_classifier.pipeline import PosturePipeline
from posture_classifier.visualization import draw_results

logging.basicConfig(level=logging.INFO)

IMAGE_PATH = Path("data/example.jpg")  # <- change me
config = load_config("configs/default.yaml")
pipeline = PosturePipeline.from_config(config)

In [ ]:
image = cv2.imread(str(IMAGE_PATH))
if image is None:
    raise FileNotFoundError(f"Could not read {IMAGE_PATH}")

people = pipeline(image)
annotated = draw_results(image, people, config.visualization)
display(Image(data=cv2.imencode(".jpg", annotated)[1].tobytes()))

for i, person in enumerate(people):
    print(i, person_to_dict(person))

To reproduce the original notebook's setup (one hand-picked box, right side only),
pass the box explicitly and switch the side rule:

In [ ]:
from dataclasses import replace

right_only = PosturePipeline(
    pipeline.detector,
    pipeline.pose_model,
    replace(config, posture=replace(config.posture, side="right")),
)
# people = right_only(image, boxes=[[x1, y1, x2, y2]])

## 4. How the rules decide

1. Person box at least as wide as tall → **Lying**.
2. Otherwise measure the **hip angle** (shoulder–hip–knee, interior, 0–180°) on the
   body side whose three keypoints are most confident.
3. Angle > 165° → **Standing**, else **Sitting**. If no side has all three keypoints
   above the confidence threshold → **Unknown**.

![rules](../docs/posture_rules.png)

*Synthetic skeletons drawn with `scripts/make_rule_figure.py`; not model output.*

## 5. Limitations

- Rules are hand-tuned, not learned, and have not been evaluated on a labelled dataset.
- The box aspect-ratio test misfires for people lying along the camera axis, or for
  standing people with arms spread wide.
- Strong foreshortening (camera looking down, person facing the camera) compresses
  the 2D hip angle; a seated person facing the camera can look like they are standing.